In [1]:
from transformers import BertTokenizer, BertForSequenceClassification

/anaconda/envs/azureml_py38_PT_TF/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-11 20:56:49.671126: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Load pre-trained BERT model and tokenizer for classification
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

# Model and tokenizer are now ready for fine-tuning

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
pip install numpy requests nlpaug

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [nlpaug]2m3/4 [nlpaug]
Note: you may need to restart the kernel to use updated packages.


In [7]:
from nlpaug.augmenter.word import BackTranslationAug

In [12]:
pip install -U sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 13.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [14]:
# Initialize the backtranslation augmenter (English -> French -> English)
back_translation_aug = BackTranslationAug(from_model_name='facebook/wmt19-en-de', to_model_name='facebook/wmt19-de-en')

# Example text to augment
text = "The weather is great today."

# Perform backtranslation to create augmented text
augmented_text = back_translation_aug.augment(text)

print("Original text:", text)
print("Augmented text:", augmented_text)

Original text: The weather is great today.
Augmented text: ['The weather today is great.']


In [15]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [41]:
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
import re

In [44]:
from transformers import BertTokenizer

# Initialize the tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [45]:
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
import re

# Load the IMDB movie reviews dataset for sentiment analysis
dataset = load_dataset('imdb')['train']  # Access the 'train' split directly

# Function to clean the text
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove punctuation and special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespaces
    return text

# Apply the cleaning function to the dataset
def clean_dataset(examples):
    examples['text'] = [clean_text(text) for text in examples['text']]
    return examples

# Clean the dataset
cleaned_dataset = dataset.map(clean_dataset, batched=True)

# Convert the cleaned dataset to a pandas DataFrame for splitting
df = cleaned_dataset.to_pandas()

# Split the DataFrame into training and validation sets
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# Convert back to Dataset if needed
train_data = Dataset.from_pandas(train_df)
val_data = Dataset.from_pandas(val_df)

# Initialize the tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Define the tokenization function
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

# Apply the tokenization function to the datasets
tokenized_train = train_data.map(tokenize_function, batched=True)
tokenized_val = val_data.map(tokenize_function, batched=True)

# Now you have tokenized_train and tokenized_val ready for model training

Map: 100%|██████████| 5000/5000 [00:23<00:00, 215.85 examples/s]
